# Week 8 activity · The Tube, three ways

**LDSCI6253 Information Presentation & Data Visualisation** · Northeastern University London

In the tutorial notebook you built networks and trees step by step. Now use the same moves on new questions
about the London Underground in Zone 1. You will:

- **A.** read the data, and find out what a row really is;
- **B.** count **degree** two ways, and see that "the busiest station" depends on what you count;
- **C.** draw the Tube as **dots and lines**, sized by how many lines stop there;
- **D.** turn the Tube into a network of **lines**, and draw it as a **grid** (matrix);
- **E.** test whether the Tube is a **tree**, and see what goes wrong when you pretend it is;
- **F.** choose the right picture for three readers, and justify it.

**How to work.** Run each cell in order (Shift + Enter). Where you see `___`, replace it with code, then run
the cell. Write your answers in the ✏️ cells. Work in pairs: one types, one checks, and swap at each part.

**Time:** about 75 minutes. Parts A–D are the core; E and F if you have time (or at home).

## 0 · Set up

The two data files (`tube_stations.csv`, `tube_links.csv`) should sit beside this notebook. If they are not
there, `load()` fetches them from the course site.

In [ ]:
import altair as alt
import pandas as pd
from pathlib import Path

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week8/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

stations = load('tube_stations')   # name, lon, lat, zone: one row per station
links = load('tube_links')         # source, target, line, colour: one row per ...?

## A · What is a row? (10 min)

**A1.** How many stations are there, and how many rows are in `links`? Look at the first rows of `links`.

In [ ]:
print(len(stations), 'stations')
print(len(___), 'rows in links')
links.head(8)

**A2.** If every row were a different piece of track, there would be one row per pair of neighbouring
stations. Keep only the first row for each `source`–`target` pair and count again.

In [ ]:
pairs = links.drop_duplicates(subset=[___])
len(pairs)

**A3.** The two numbers differ. Find a pair of stations that appears more than once, and say why.
*Hint:* `links[links.duplicated(subset=['source', 'target'], keep=False)].head(6)`

In [ ]:
___

✏️ *In one sentence: what does one row of `links` stand for?*

## B · Degree, two ways (10 min)

In the tutorial, **degree** was how many edges a node has. On the Tube we can count two different things:

- **edges**: every row in `links` that touches the station (so shared track counts once per line);
- **neighbours**: how many *different* stations you can reach in one stop (count in `pairs`).

**B1.** Count both, put them side by side, and sort by neighbours.

In [ ]:
by_edges = pd.concat([links['source'], links['target']]).value_counts()
by_neighbours = pd.concat([___]).value_counts()

degree = pd.DataFrame({'edges': by_edges, 'neighbours': by_neighbours})
degree.sort_values(___, ascending=False).head(8)

**B2.** Now sort by `edges` instead. Is the "busiest" station the same?

In [ ]:
degree.sort_values(___, ascending=False).head(8)

✏️ *Which station is busiest by edges, and which by neighbours? Why do the answers differ?*

## C · Dots and lines: where the lines meet (15 min)

**C1.** Make a table of which lines serve each station (one row per station and line), then count the lines
at each station.

In [ ]:
served = pd.concat([
    links[['source', 'line', 'colour']].rename(columns={'source': 'station'}),
    links[['target', 'line', 'colour']].rename(columns={'target': 'station'}),
]).drop_duplicates()

lines_at = served.groupby('station')['line'].count().rename(___)
lines_at.sort_values(ascending=False).head(6)

**C2.** Draw the Tube as in the tutorial (section 4), but **size each station by how many lines stop there**,
and name the stations with five or more lines (more names would pile up in the centre). Give the chart a title and a source.

In [ ]:
dots_table = stations.merge(lines_at, left_on='name', right_index=True)

# The same join as the tutorial: each edge gets both of its ends
track = (links
    .merge(stations, left_on='source', right_on='name')
    .merge(stations, left_on='target', right_on='name', suffixes=('', '2')))

where = dict(x=alt.X('lon:Q', scale=alt.Scale(zero=False), axis=None),
             y=alt.Y('lat:Q', scale=alt.Scale(zero=False), axis=None))

rails = alt.Chart(track).mark_rule(strokeWidth=3).encode(
    **where, x2='lon2:Q', y2='lat2:Q', color=alt.Color('colour:N', scale=None))
dots = alt.Chart(dots_table).mark_circle(color='white', stroke='black', opacity=1).encode(
    **where, size=alt.Size(___, title='Lines stopping here', scale=alt.Scale(domain=[1, 6]),
                  legend=alt.Legend(values=[1, 2, 3, 4, 5, 6])),
    tooltip=['name:N', 'lines:Q'])
names = alt.Chart(dots_table).mark_text(dy=-14, fontWeight='bold', fontSize=11).encode(
    **where, text='name:N').transform_filter(___)

(rails + dots + names).properties(
    width=680, height=440,
    title=alt.Title('Zone 1: where the Tube lines meet', subtitle='Source: TfL Open Data')
).configure_view(stroke=None)

✏️ *What does position mean in this picture? What does size mean? Name one thing this picture is bad at showing.*

## D · A network of lines, as a grid (20 min)

Turn the question round: the **nodes are now the 11 lines**, and two lines are linked when they share a
station. The more stations they share, the stronger the link.

**D1.** Pair every line at a station with every other line at that station, then count the stations each pair
of lines shares.

In [ ]:
both = served.merge(served, on=___)      # every pair of lines that meet at a station
both = both[both['line_x'] != both['line_y']]      # a line does not link to itself

shared = both.groupby(['line_x', 'line_y']).size().reset_index(name='stations')
shared.sort_values('stations', ascending=False).head(6)

**D2.** Draw it as a **matrix**: one row and one column per line, the colour of a square for how many stations
they share, and the number written in it. Order the rows from the line with the most stations to the fewest.

In [ ]:
order = served['line'].value_counts().index.tolist()   # most stations first

cells = alt.Chart(shared).encode(
    x=alt.X('line_x:N', sort=order, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('line_y:N', sort=order, title=None))
squares = cells.mark_rect().encode(
    color=alt.Color(___, scale=alt.Scale(scheme='blues'), title='Stations shared'),
    tooltip=['line_x:N', 'line_y:N', 'stations:Q'])
numbers = cells.mark_text(fontSize=11).encode(
    text='stations:Q',
    color=alt.condition('datum.stations > 8', alt.value('white'), alt.value('black')))   # light on dark

(squares + numbers).properties(
    width=400, height=400,
    title=alt.Title('Which Tube lines share stations in Zone 1?', subtitle='Source: TfL Open Data')
).configure_view(stroke=None)

✏️ *Which two lines share the most stations? Why is a grid a better picture than dots and lines for this network?*

## E · Is the Tube a tree? (10 min)

A **tree** needs every child to have exactly **one** parent. Suppose we try Tube → line → station.

**E1.** How many stations have more than one line? (Use `lines_at` from C1.)

In [ ]:
print((lines_at ___).sum(), 'of', len(lines_at), 'stations are on more than one line')
lines_at.value_counts().sort_index().rename('stations')   # how many stations have 1, 2, 3 ... lines

**E2.** Pretend it is a tree anyway, and draw an icicle as in the tutorial (section 6c): one bar per level,
counting stations. Compare the length of the Station bar with the number of stations.

In [ ]:
rows = pd.concat([
    served.assign(level='1 Tube', name='Tube', colour='#888888'),
    served.assign(level='2 Line', name=served['line']),
    served.assign(level='3 Station', name=served['station']),
])

alt.Chart(rows).mark_bar(stroke='white').encode(
    y=alt.Y('level:O', title=None),
    x=alt.X(___, title='Stations counted'),
    detail='name:N',
    color=alt.Color('colour:N', scale=None),
    tooltip=['name:N', 'count():Q'],
).properties(width=640, height=180, title='The Tube forced into a tree')

In [ ]:
print(len(rows[rows['level'] == '3 Station']), 'station boxes, but only', len(stations), 'stations')

✏️ *Is the Tube a tree? What goes wrong in the icicle when we pretend it is?*

## F · Choose and justify (10 min)

For each reader, choose **dots and lines**, **a grid**, or **boxes / bars inside each other**, and give one
reason. Use the pictures you made.

1. A tourist asks: *"How do I get from Paddington to Bank?"*
2. TfL planners ask: *"If the Circle line closes, which other lines lose the most interchanges?"*
3. A council asks: *"Which areas of London have the most people, and which boroughs make them up?"*

✏️ *1: … because …  2: … because …  3: … because …*

**F2 (extension).** Pick one of your pictures and improve it for the tourist or the planner: a clearer title
that says what to see, a key, fewer labels, or a highlighted route. Explain your change in one sentence.

In [ ]:
# Your improved picture here